In [ ]:
from pathlib import Path

import pandas as pd

from config.config import REGISTRIES_DIR, SUBREGISTRIES_DIR, VersionObject

DOCUMENT_REGISTRY: Path =  REGISTRIES_DIR / "document_registry.csv"
BASE_REGISTRY: Path = REGISTRIES_DIR / "base_output_registry.csv"
RAW_REGISTRY: Path = SUBREGISTRIES_DIR / "raw_file_output_registry.csv"
EXTRACTION_REGISTRY: Path = SUBREGISTRIES_DIR / "extraction_output_registry.csv"
CAS_CLASSIFICATION_REGISTRY: Path = SUBREGISTRIES_DIR / "CAS_classification_output_registry.csv"

version_object = VersionObject(pipeline_version="v1.0.0", software_version="v1.0.4")
df_document_registry = pd.read_csv(DOCUMENT_REGISTRY)
df_base_registry = pd.read_csv(BASE_REGISTRY)
df_raw_registry = pd.read_csv(RAW_REGISTRY)
df_extraction_registry = pd.read_csv(EXTRACTION_REGISTRY)
df_CAS_classification_registry = pd.read_csv(CAS_CLASSIFICATION_REGISTRY)

In [ ]:
merge_base_CAS_classification = df_CAS_classification_registry.merge(df_base_registry[["output_sha", "doc_doi"]], on="output_sha", how="left")

df_classification_reduced = merge_base_CAS_classification[["output_sha", "doc_doi", "text"]]


In [ ]:
display(df_classification_reduced)

In [ ]:
df_CAS_sample = df_classification_reduced.sample(n=500, random_state=42)

df_CAS_sample.to_csv("validation_sample_CAS.csv", index=False)

In [ ]:
df_CAS_manual = pd.read_csv("data/metadata/annotations/CAS_annotated_sample.csv")

df_CAS_manual_short = df_CAS_manual[["output_sha", "doc_doi", "mca_label", "eta_label", "text"]].rename(columns={"mca_label": "mca_label_manual", "eta_label": "eta_label_manual"})

df_CAS_classifier = df_CAS_classification_registry[df_CAS_classification_registry["output_sha"].isin(df_CAS_manual_short["output_sha"])][["output_sha", "MCA_label", "ETA_label"]].rename(columns={"MCA_label": "mca_label_classifier", "ETA_label": "eta_label_classifier"})


In [ ]:

df_CAS_validation = df_CAS_manual_short.merge(df_CAS_classifier, on="output_sha", how="left")

# here loading the existing one with one item manually corrected

df_CAS_validation = pd.read_csv("data/metadata/annotations/CAS/validated_sample_CAS.csv")

print(f"Matched rows: {len(df_CAS_validation)}")

In [ ]:
"""TESTING CAS MANUAL-CLASSIFIER AGREEMENT WITH COHEN'S KAPPA"""

from sklearn.metrics import cohen_kappa_score

kappa_MCA = cohen_kappa_score(df_CAS_validation["mca_label_manual"], df_CAS_validation["mca_label_classifier"])

kappa_ETA = cohen_kappa_score(df_CAS_validation["eta_label_manual"], df_CAS_validation["eta_label_classifier"])

print(df_CAS_validation[["mca_label_manual", "mca_label_classifier"]].value_counts())

print(df_CAS_validation[["eta_label_manual", "eta_label_classifier"]].value_counts())

print(f"\nCohen's Kappa for MCA classification: {kappa_MCA:.4f}")

print(f"\nCohen's Kappa for ETA classification: {kappa_ETA:.4f}")

In [ ]:
"""CONFUSION MATRIX FOR CAS MANUAL-CLASSIFIER AGREEMENTS"""

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

labels = sorted(df_CAS_validation["mca_label_manual"].unique())
cm = confusion_matrix(df_CAS_validation["mca_label_manual"], 
                      df_CAS_validation["mca_label_classifier"], 
                      labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(xticks_rotation=45)
plt.tight_layout()
plt.savefig("data/images/CAS/mca_confusion_matrix_validation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
"""CONFUSION MATRIX FOR CAS MANUAL-CLASSIFIER AGREEMENTS"""

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

labels = sorted(df_CAS_validation["eta_label_manual"].unique())
cm = confusion_matrix(df_CAS_validation["eta_label_manual"], 
                      df_CAS_validation["eta_label_classifier"], 
                      labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(xticks_rotation=45)
plt.tight_layout()
plt.savefig("data/images/CAS/mca_confusion_matrix_validation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df_MCA_disagreements = df_CAS_validation[df_CAS_validation["mca_label_manual"] != df_CAS_validation["mca_label_classifier"]]

total_disagreeements = len(df_MCA_disagreements)

print(f"Total disagreements: {total_disagreeements}\n")

for _, row in df_MCA_disagreements.iterrows():

    print(f"DOI:            {row['doc_doi']}")

    print(f"Output_sha:     {row['output_sha']}")

    print(f"Manual:         {row['mca_label_manual']}")

    print(f"Classifier:     {row['mca_label_classifier']}")

    print(f"Text:           {row['text']}")

In [ ]:
df_ETA_disagreements = df_CAS_validation[df_CAS_validation["eta_label_manual"] != df_CAS_validation["eta_label_classifier"]]

total_disagreeements_eta = len(df_ETA_disagreements)

print(f"Total disagreements: {total_disagreeements_eta}\n")

for _, row in df_ETA_disagreements.iterrows():

    print(f"DOI:            {row['doc_doi']}")

    print(f"Output_sha:     {row['output_sha']}")

    print(f"Manual:         {row['eta_label_manual']}")

    print(f"Classifier:     {row['eta_label_classifier']}")

    print(f"Text:           {row['text']}")

In [ ]:
import numpy as np
import scipy.stats as stats


def kappa_ci(kappa, n, alpha=0.05):
    se = np.sqrt((kappa * (1 - kappa)) / n)
    z = stats.norm.ppf(1 - alpha / 2)
    return kappa - z * se, kappa + z * se

mca_lower, mca_upper = kappa_ci(kappa_MCA, 500)
print(f"95% CI: [{mca_lower:.4f}, {mca_upper:.4f}]")

In [ ]:


def kappa_ci(kappa, n, alpha=0.05):
    se = np.sqrt((kappa * (1 - kappa)) / n)
    z = stats.norm.ppf(1 - alpha / 2)
    return kappa - z * se, kappa + z * se
eta_lower, eta_upper = kappa_ci(kappa_ETA, 500)
print(f"95% CI: [{eta_lower:.4f}, {eta_upper:.4f}]")

In [ ]:
df_CAS_validation.to_csv("data/metadata/annotations/validated_sample_CAS.csv")